# CartRank: fine-tune bge-small on ESCI (Kaggle)

Fine-tunes `BAAI/bge-small-en-v1.5` as a bi-encoder with `MultipleNegativesRankingLoss` on ESCI
(query, Exact product) pairs. Run on a **T4** with **Internet ON**. fp16 only (bf16 is not
supported on T4/P100). No tokens or secrets are needed: the model and data are public.

The notebook rebuilds the exact same dev split as the local repo (10% of train query_ids, seed 0)
and **asserts** it, so dev queries never leak into training.

Outputs in `/kaggle/working`: `bge-small-esci-ft/` (final model), `bge-small-esci-ft.zip`,
`train_log.json`, `dev_check.json`.

In [ ]:
!pip install -q -U sentence-transformers pyarrow
import sentence_transformers
import torch

print(sentence_transformers.__version__, torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
# ---- Config ---------------------------------------------------------------------------
BASE_MODEL = "BAAI/bge-small-en-v1.5"
QUERY_PREFIX = "Represent this sentence for searching relevant passages: "  # bge model card
MAX_SEQ_LENGTH = 128
EPOCHS = 1  # try 2 for a second run; keep run names distinct
BATCH_SIZE = 128  # in-batch negatives: larger is better if memory allows
LR = 5e-5
USE_HARD_NEG = False  # True: (query, Exact, Irrelevant) triplets, only queries that have an I
RUN_NAME = "bge-small-esci-ft"
OUT_DIR = f"/kaggle/working/{RUN_NAME}"
SEED = 0
# Checksum of the local dev query ids (computed in the repo); must match.
EXPECTED_DEV_QUERIES = 2089
EXPECTED_DEV_SHA = "9815cfa99a926133"

In [ ]:
import os
import urllib.request

BASE = "https://github.com/amazon-science/esci-data/raw/main/shopping_queries_dataset/"
os.makedirs("/kaggle/working/esci", exist_ok=True)
for f in ["shopping_queries_dataset_examples.parquet", "shopping_queries_dataset_products.parquet"]:
    dst = f"/kaggle/working/esci/{f}"
    if not os.path.exists(dst):
        urllib.request.urlretrieve(BASE + f, dst)
    print(f, os.path.getsize(dst) // 10**6, "MB")

In [ ]:
import hashlib

import numpy as np
import pandas as pd

ex = pd.read_parquet("/kaggle/working/esci/shopping_queries_dataset_examples.parquet")
ex = ex[(ex.product_locale == "us") & (ex.small_version == 1)]
train_all = ex[ex.split == "train"]

# Same rule as cartrank.data.splits.split_queries(dev_frac=0.1, seed=0)
ids = np.sort(train_all.query_id.unique())
rng = np.random.default_rng(SEED)
dev_ids = set(rng.choice(ids, size=int(round(len(ids) * 0.1)), replace=False).tolist())
sha = hashlib.sha256(",".join(map(str, sorted(dev_ids))).encode()).hexdigest()[:16]
assert len(dev_ids) == EXPECTED_DEV_QUERIES and sha == EXPECTED_DEV_SHA, (len(dev_ids), sha)
train = train_all[~train_all.query_id.isin(dev_ids)]
dev = train_all[train_all.query_id.isin(dev_ids)]
assert not set(train.query_id) & set(dev.query_id)
print("train queries", train.query_id.nunique(), "dev queries", dev.query_id.nunique())

In [ ]:
# Product text, identical to cartrank.search.corpus.product_text(variant="tbcb")
prod = pd.read_parquet(
    "/kaggle/working/esci/shopping_queries_dataset_products.parquet",
    columns=[
        "product_id",
        "product_locale",
        "product_title",
        "product_brand",
        "product_color",
        "product_bullet_point",
    ],
)
prod = prod[
    (prod.product_locale == "us") & prod.product_id.isin(set(ex.product_id))
].drop_duplicates("product_id")


def clean(s):
    return " ".join(s.split()) if isinstance(s, str) else ""


def text(r):
    parts = [
        clean(r.product_title),
        clean(r.product_brand),
        clean(r.product_color),
        clean(r.product_bullet_point)[:500],
    ]
    return ". ".join(p for p in parts if p)


prod_text = {r.product_id: text(r) for r in prod.itertuples()}
print(len(prod_text), "products;", next(iter(prod_text.values()))[:200])

In [ ]:
from datasets import Dataset

pos = train[train.esci_label == "E"][["query_id", "query", "product_id"]]
rows = []
if USE_HARD_NEG:
    neg_pool = (train[train.esci_label == "I"].groupby("query_id").product_id.apply(list)).to_dict()
    rnd = np.random.default_rng(SEED)
    for r in pos.itertuples():
        negs = neg_pool.get(r.query_id)
        if negs:
            rows.append(
                {
                    "anchor": QUERY_PREFIX + r.query,
                    "positive": prod_text[r.product_id],
                    "negative": prod_text[negs[rnd.integers(len(negs))]],
                }
            )
else:
    for r in pos.itertuples():
        rows.append({"anchor": QUERY_PREFIX + r.query, "positive": prod_text[r.product_id]})
train_ds = Dataset.from_list(rows).shuffle(seed=SEED)
print(len(train_ds), "training examples;", train_ds[0])

In [ ]:
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
    losses,
)
from sentence_transformers.training_args import BatchSamplers

model = SentenceTransformer(BASE_MODEL, device="cuda")
model.max_seq_length = MAX_SEQ_LENGTH
loss = losses.MultipleNegativesRankingLoss(model)

args = SentenceTransformerTrainingArguments(
    output_dir=f"/kaggle/working/ckpt-{RUN_NAME}",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LR,
    warmup_ratio=0.1,
    fp16=True,
    bf16=False,  # T4/P100: fp16 only
    batch_sampler=BatchSamplers.NO_DUPLICATES,  # no repeated query/product in a batch (false negatives)
    save_strategy="steps",
    save_steps=500,
    save_total_limit=2,  # checkpoints to /kaggle/working
    logging_steps=50,
    report_to="none",
    seed=SEED,
)
trainer = SentenceTransformerTrainer(model=model, args=args, train_dataset=train_ds, loss=loss)

In [ ]:
import glob
import json

ckpts = sorted(glob.glob(f"{args.output_dir}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
trainer.train(
    resume_from_checkpoint=ckpts[-1] if ckpts else None
)  # resumes after a session restart
model.save(OUT_DIR)
json.dump(trainer.state.log_history, open("/kaggle/working/train_log.json", "w"), indent=1)

## Quick dev check (rerank mode, same graded NDCG@10 as the repo)
A sanity check only. The reported numbers come from the repo's evaluation code after the model is
downloaded and run locally.

In [ ]:
import math

GAINS = {"E": 1.0, "S": 0.1, "C": 0.01, "I": 0.0}


def ndcg10(ranked, labels):
    dcg = lambda g: sum(x / math.log2(i + 2) for i, x in enumerate(g[:10]))
    gains = [GAINS[labels[p]] for p in ranked]
    ideal = sorted((GAINS[v] for v in labels.values()), reverse=True)
    return dcg(gains) / dcg(ideal) if dcg(ideal) > 0 else 0.0


def dev_ndcg(m):
    m.max_seq_length = MAX_SEQ_LENGTH
    pids = sorted(set(dev.product_id))
    pv = m.encode([prod_text[p] for p in pids], batch_size=256, normalize_embeddings=True)
    row = {p: i for i, p in enumerate(pids)}
    qs = dev.drop_duplicates("query_id")
    qv = m.encode(
        [QUERY_PREFIX + q for q in qs["query"]], batch_size=256, normalize_embeddings=True
    )
    scores = []
    for i, qid in enumerate(qs.query_id):
        g = dev[dev.query_id == qid]
        labels = dict(zip(g.product_id, g.esci_label))
        s = pv[[row[p] for p in labels]] @ qv[i]
        ranked = [list(labels)[j] for j in np.argsort(-s, kind="stable")]
        scores.append(ndcg10(ranked, labels))
    return float(np.mean(scores))


base = SentenceTransformer(BASE_MODEL, device="cuda")
ft = SentenceTransformer(OUT_DIR, device="cuda")
check = {
    "base_ndcg10_rerank": dev_ndcg(base),
    "ft_ndcg10_rerank": dev_ndcg(ft),
    "epochs": EPOCHS,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "hard_neg": USE_HARD_NEG,
    "train_examples": len(train_ds),
}
json.dump(check, open("/kaggle/working/dev_check.json", "w"), indent=1)
print(check)

In [ ]:
import shutil

shutil.make_archive(f"/kaggle/working/{RUN_NAME}", "zip", OUT_DIR)
print(os.path.getsize(f"/kaggle/working/{RUN_NAME}.zip") // 10**6, "MB zip")